# tiny-ton on a Real GPU

This notebook builds **tiny-ton** from source with CUDA enabled and runs `vector_add` on the Colab T4 GPU.

**Before running:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

## 1. Install LLVM/MLIR 18 + pybind11

Uses pre-built packages from `apt.llvm.org` (~2 min).

In [11]:
%%bash
set -e
echo '>>> Adding LLVM 18 apt repository...'
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
add-apt-repository -y 'deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main' > /dev/null 2>&1
echo '>>> Installing packages...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build > /dev/null 2>&1
pip install -q pybind11
echo '>>> Done. MLIR version:'
mlir-opt-18 --version 2>&1 | head -2

>>> Adding LLVM 18 apt repository...
>>> Installing packages...
>>> Done. MLIR version:
Ubuntu LLVM version 18.1.8
  Optimized build.


## 2. Clone tiny-ton

In [12]:
%%bash
set -e
if [ ! -d /content/tiny-ton ]; then
  git clone https://github.com/ganeshnj/tiny-ton.git /content/tiny-ton
else
  echo 'tiny-ton already cloned, pulling latest...'
  cd /content/tiny-ton && git pull
fi

tiny-ton already cloned, pulling latest...
Already up to date.


## 3. Build with CUDA enabled

Compiles the C++ compiler, MLIR lowering passes, CUDA runtime, and Python bindings (~1-2 min).

In [13]:
%%bash
set -e
cd /content/tiny-ton
rm -rf build
cmake -G Ninja -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \
  -DTTN_ENABLE_CUDA=ON \
  -DTTN_ENABLE_PYTHON=ON \
  -DCUDAToolkit_ROOT=/usr/local/cuda \
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
echo '>>> Build complete.'

-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Performing Test HAVE_FFI_CALL
-- Performing Test HAVE_FFI_CALL - Success
-- Found FFI: /usr/lib/x86_64-linux-gnu/libffi.so
-- Could NOT find LibEdit (missing: LibEdit_INCLUDE_DIRS LibEdit_LIBRARIES) 
-- Performing Test Terminfo_LINKABLE
-- Performing Test Terminfo_LINKABLE - Success
-- Found Terminfo: /usr/lib/x86_64-linux-gnu/libtinfo.so
-- Found ZLIB: /usr/lib/x86_64-linux-gnu/libz.so (found version "1.2.11")
-- Found zstd: /usr/lib/x86_64-linux-gnu/libzstd.so
-- Found LibXml2: /usr/li

CMake Warning (dev) at /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11/FindPythonLibsNew.cmake:101 (message):
  Policy CMP0148 is not set: The FindPythonInterp and FindPythonLibs modules
  are removed.  Run "cmake --help-policy CMP0148" for policy details.  Use
  the cmake_policy command to set the policy and suppress this warning, or
  preferably upgrade to using FindPython, either by calling it explicitly
  before pybind11, or by setting PYBIND11_FINDPYTHON ON before pybind11.
Call Stack (most recent call first):
  /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11/pybind11Tools.cmake:44 (find_package)
  /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11/pybind11Common.cmake:237 (include)
  /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11/pybind11Config.cmake:257 (include)
  bindings/CMakeLists.txt:1 (find_package)
This warning is for project developers.  Use -Wno-dev to suppress it.



## 4. Verify GPU

In [14]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
import importlib
importlib.invalidate_caches()
import _tiny_ton_core as core
print(f'has_cuda() = {core.has_cuda()}')

has_cuda() = True


## 5. Debug pipeline (stage-by-stage IR dump)

Shows every compilation stage: Python source -> AST -> TinyTon MLIR -> simulator assembly -> PTX.

In [15]:
import os
os.environ['TTN_SM_VERSION'] = 'sm_75'

!cd /content/tiny-ton && TTN_SM_VERSION=sm_75 \
  PYTHONPATH=build/bindings:python \
  python3 examples/debug_pipeline.py


  STAGE 0: Python Source

@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)


  STAGE 1: Python AST

Module(
  body=[
    FunctionDef(
      name='vector_add',
      args=arguments(
        posonlyargs=[],
        args=[
          arg(arg='a_ptr'),
          arg(arg='b_ptr'),
          arg(arg='c_ptr'),
          arg(arg='N')],
        kwonlyargs=[],
        kw_defaults=[],
        defaults=[]),
      body=[
        Assign(
          targets=[
            Name(id='pid', ctx=Store())],
          value=Call(
            func=Attribute(
              value=Name(id='tt', ctx=Load()),
              attr='program_id',
              ctx=Load()),
            args=[
              Constant(value=0)],
            keywords=[])),
        Assign(
          targets=[
          

## 6. Run vector_add on the real GPU

Full end-to-end: Python -> TinyTon MLIR -> PTX -> CUDA driver API -> T4 GPU -> results back.

In [16]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')
os.environ['TTN_SM_VERSION'] = 'sm_75'

import numpy as np
import tiny_ton as tt

@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)

N = 256
a = np.arange(N, dtype=np.int32)
b = np.arange(N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

grid = (N // 64,)
vector_add[grid](a, b, c, N)

print(f'a[:8]      = {a[:8]}')
print(f'b[:8]      = {b[:8]}')
print(f'c[:8]      = {c[:8]}')
print(f'expected   = {(a+b)[:8]}')

assert np.array_equal(c, a + b), 'MISMATCH!'
print('\nPASS -- vector_add ran on the T4 GPU!')

a[:8]      = [0 1 2 3 4 5 6 7]
b[:8]      = [0 1 2 3 4 5 6 7]
c[:8]      = [ 0  2  4  6  8 10 12 14]
expected   = [ 0  2  4  6  8 10 12 14]

PASS -- vector_add ran on the T4 GPU!


## 7. Run vector_add with f32 on the GPU

Same kernel, float32 arrays -- type is inferred from numpy dtype.

In [17]:
N = 256
a_f = np.random.randn(N).astype(np.float32)
b_f = np.random.randn(N).astype(np.float32)
c_f = np.zeros(N, dtype=np.float32)

vector_add[(N // 64,)](a_f, b_f, c_f, N)

print(f'a[:8]      = {a_f[:8]}')
print(f'b[:8]      = {b_f[:8]}')
print(f'c[:8]      = {c_f[:8]}')
print(f'expected   = {(a_f+b_f)[:8]}')

assert np.allclose(c_f, a_f + b_f, atol=1e-6), 'f32 MISMATCH!'
print('\nPASS -- f32 vector_add ran on the T4 GPU!')

a[:8]      = [ 0.5907042   0.11529873  0.02964292  2.9586256  -0.00612996 -0.15924521
 -0.12144867 -0.58353674]
b[:8]      = [ 1.9308509  -0.40584338 -1.0885086   0.5685042   0.14993383  0.5496426
  0.30378792  0.24376576]
c[:8]      = [ 2.521555   -0.29054466 -1.0588657   3.5271297   0.14380386  0.39039743
  0.18233925 -0.33977097]
expected   = [ 2.521555   -0.29054466 -1.0588657   3.5271297   0.14380386  0.39039743
  0.18233925 -0.33977097]

PASS -- f32 vector_add ran on the T4 GPU!


## 8. Run vector_add with f16 on the GPU

Half-precision float16 -- same kernel, type inferred from `np.float16` arrays. T4 supports native f16 arithmetic (`add.rn.f16` in PTX).

In [18]:
N = 256
a_h = np.random.randn(N).astype(np.float16)
b_h = np.random.randn(N).astype(np.float16)
c_h = np.zeros(N, dtype=np.float16)

vector_add[(N // 64,)](a_h, b_h, c_h, N)

expected_h = (a_h + b_h).astype(np.float16)
print(f'a[:8]      = {a_h[:8]}')
print(f'b[:8]      = {b_h[:8]}')
print(f'c[:8]      = {c_h[:8]}')
print(f'expected   = {expected_h[:8]}')

assert np.allclose(c_h, expected_h, atol=1e-2), 'f16 MISMATCH!'
print('\nPASS -- f16 vector_add ran on the T4 GPU!')

a[:8]      = [-2.236  -1.095   2.332   0.3452  0.0528 -1.621  -0.2456  0.0481]
b[:8]      = [ 0.1242  1.183   0.943   0.203  -0.7354  0.1864  1.085   1.191 ]
c[:8]      = [-2.111   0.0879  3.275   0.5483 -0.6826 -1.435   0.8394  1.239 ]
expected   = [-2.111   0.0879  3.275   0.5483 -0.6826 -1.435   0.8394  1.239 ]

PASS -- f16 vector_add ran on the T4 GPU!


## 9. Math intrinsics on the GPU

Test `exp`, `log`, `sqrt`, `rsqrt`, `abs`, `max` -- the building blocks of softmax, rmsnorm, and loss.

In [19]:
@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.exp(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_log(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.log(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_sqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.sqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_rsqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.rsqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_abs(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.abs(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_max(a_ptr, b_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(dst + off, tt.max(a, b), mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(42)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-5), ('f16', np.float16, 5e-2)]:
    print(f'\n--- {dtype_name} ---')
    pos = (np.abs(np.random.randn(N)) + 0.01).astype(dtype)
    mixed = (np.random.randn(N) * 2).astype(dtype)

    for name, kernel, np_fn, inp in [
        ('exp',  kern_exp,  np.exp,  mixed),
        ('log',  kern_log,  np.log,  pos),
        ('sqrt', kern_sqrt, np.sqrt, pos),
        ('abs',  kern_abs,  np.abs,  mixed),
    ]:
        dst = np.zeros(N, dtype=dtype)
        kernel[grid](inp.copy(), dst, N)
        ok = np.allclose(dst, np_fn(inp), atol=atol, rtol=1e-2)
        print(f'  {name}: {"PASS" if ok else "FAIL"}')

    # rsqrt
    dst = np.zeros(N, dtype=dtype)
    kern_rsqrt[grid](pos.copy(), dst, N)
    expected_rsqrt = (1.0 / np.sqrt(pos.astype(np.float32))).astype(dtype)
    ok = np.allclose(dst, expected_rsqrt, atol=atol, rtol=1e-2)
    print(f'  rsqrt: {"PASS" if ok else "FAIL"}')

    # max (binary)
    a = (np.random.randn(N) * 5).astype(dtype)
    b = (np.random.randn(N) * 5).astype(dtype)
    dst = np.zeros(N, dtype=dtype)
    kern_max[grid](a.copy(), b.copy(), dst, N)
    ok = np.allclose(dst, np.maximum(a, b), atol=atol, rtol=1e-2)
    print(f'  max: {"PASS" if ok else "FAIL"}')


--- f32 ---
  exp: PASS
  log: PASS
  sqrt: PASS
  abs: PASS
  rsqrt: PASS
  max: PASS

--- f16 ---
  exp: PASS
  log: PASS
  sqrt: PASS
  abs: PASS
  rsqrt: PASS
  max: PASS


## 10. Reductions on the GPU

Test `tt.reduce_sum` and `tt.reduce_max` — block-level reductions via `gpu.all_reduce` (warp-shuffle under the hood). Each thread contributes one value; every thread receives the reduced scalar.

In [20]:
from typing import Any


@tt.jit
def kern_reduce_sum(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    total = tt.reduce_sum(x)
    tt.store(dst + pid, total)

@tt.jit
def kern_reduce_max(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    x = tt.load(src + off, mask=mask)
    mx = tt.reduce_max(x)
    tt.store(dst + pid, mx)

N = 256
BLOCK = 64
grid = (N // BLOCK,)
np.random.seed(123)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-4)]:
    print(f'\n--- {dtype_name} ---')
    x = np.random.randn(N).astype(dtype)

    # reduce_sum
    out_sum = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_sum[grid](x.copy(), out_sum, N)
    for i in range(N // BLOCK):
        expected = np.sum(x[i*BLOCK:(i+1)*BLOCK])
        ok = np.allclose(out_sum[i], expected, atol=atol)
        print(f'  reduce_sum block {i}: {"PASS" if ok else "FAIL"}')

    # reduce_max
    out_max = np.zeros(N // BLOCK, dtype=dtype)
    kern_reduce_max[grid](x.copy(), out_max, N)
    for i in range(N // BLOCK):
        expected = np.max(x[i*BLOCK:(i+1)*BLOCK])
        ok = np.allclose(out_max[i], expected, atol=atol)
        print(f'  reduce_max block {i}: {"PASS" if ok else "FAIL"}')


--- f32 ---


AssertionError: NVPTX compilation failed: MLIR pass pipeline failed: failed to legalize operation 'gpu.all_reduce' that was explicitly marked illegal
  note: see current operation: 
%0 = "gpu.all_reduce"(<<UNKNOWN SSA VALUE>>) <{op = #gpu<all_reduce_op add>}> ({
}) : (f32) -> f32
